# I09 - Advanced Transformer Architectures

**Interactive Application:** [Transformer Explorer](https://nexageapps.github.io/AI/compsci714/week4/transformer-explorer/) - Explore self-attention, multi-head attention, and compare encoder/decoder architectures interactively!

**MAI Alignment:** COMPSCI 714 (Lectures 8-1, 8-2), COMPSCI 703, COMPSYS 721 | **Prerequisite:** [B11 - Attention and Transformers](../Basic/B11%20-%20Attention%20and%20Transformers.ipynb) | **Next:** [I05b - Pretrained Foundation Models in CV](./I05b%20-%20Pretrained%20Foundation%20Models%20in%20CV.ipynb)

**Level:** Intermediate

---

## Learning Objectives

By the end of this lesson, you will:
1. Understand the **Pre-training and Fine-tuning Paradigm** and why it works (AI Scaling Law)
2. Understand **Pretrained Foundation Models (PFMs)** and their role
3. Master **BERT** architecture: Masked Language Modeling (MLM) and Next Sentence Prediction (NSP)
4. Understand **T5** (Text-To-Text Transfer Transformer) and Span Corruption pretraining
5. Understand **GPT** (decoder-only) and Causal Language Modeling
6. Compare all three architectures: Encoders, Encoder-Decoders, and Decoders
7. Know when to use each architecture for different NLP tasks

---

## Prerequisites

- Completed B11 (Attention and Transformers)
- Understanding of self-attention
- Familiarity with transformer basics
- Completed I05 (Transfer Learning) — especially self-supervised learning

---

> 📚 **COMPSCI 714 Alignment:** This notebook directly covers Lectures 8-1 (Pretrained Foundation Models in NLP). The pre-training paradigm, BERT, T5, and GPT sections match the course slides exactly.

In [ ]:
# Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f"TensorFlow version: {tf.__version__}")

## 1. Pre-training and Fine-tuning Paradigm (COMPSCI 714 Lecture 8-1)

### The Core Paradigm

From COMPSCI 714 Lecture 8-1:

| Stage | Data | Process |
|-------|------|---------|
| **Pre-training** | Large, general, **unlabelled** | Model learns general-purpose knowledge in a self-supervised way |
| **Fine-tuning** | Small, task-specific, **labelled** | Adjust pre-trained model for a specific downstream task |

The pre-trained models are called **base models** or **foundational models**.

**Analogy:**
- Pre-training is like attending school and learning general knowledge
- Fine-tuning is like job training for a specific profession

### Why Pre-training? The AI Scaling Law

**AI Scaling Law** (Kaplan et al., 2020): The performance of AI models improves in a **predictable way** as you scale up:
1. **Model size** — number of parameters
2. **Dataset size** — how much data the model is trained on
3. **Compute budget** — total computation used for training (FLOPs)

**GPT parameter evolution:**
- GPT-1 (2018): 117M parameters
- GPT-2 (2019): 1.5B parameters
- GPT-3 (2020): 175B parameters
- GPT-4 (2023): Trillions of parameters

**The challenge:** Training large models on large-scale data requires **unlabelled data** — it's virtually impossible to annotate billions of documents. This is why we need **self-supervised learning**.

### Pretrained Foundation Models (PFMs)

**Definition:** PFMs are regarded as the **foundation** for various downstream tasks. A PFM is trained on large-scale data which provides a reasonable parameter initialisation for a wide range of downstream applications.

**Key ideas in pretraining:**
- Make sure your model can process large-scale, diverse datasets
- Don't use labelled data (otherwise you can't scale!)
- Compute-aware scaling

### Three Architectures for Pretraining in NLP

The neural architecture influences the type of pretraining and natural use cases:

| Architecture | Example | Pretraining Task | Best For |
|---|---|---|---|
| **Encoders** | BERT | Masked Language Modeling (MLM) | Text analysis, classification, NLU |
| **Encoder-Decoders** | T5 | Span Corruption | Seq2seq tasks (translation, summarisation) |
| **Decoders** | GPT | Causal Language Modeling (CLM) | Text generation, scale well |

In [ ]:
class BERTEncoder(layers.Layer):
    """Simplified BERT encoder layer"""
    
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='gelu'),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)
    
    def call(self, inputs, training=False):
        # Multi-head attention
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        
        # Feed-forward network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def create_bert_style_model(vocab_size=30000, max_len=512, embed_dim=768, 
                           num_heads=12, ff_dim=3072, num_layers=12):
    """Create BERT-style model
    
    Args:
        vocab_size: Vocabulary size
        max_len: Maximum sequence length
        embed_dim: Embedding dimension
        num_heads: Number of attention heads
        ff_dim: Feed-forward dimension
        num_layers: Number of encoder layers
    """
    inputs = keras.Input(shape=(max_len,))
    
    # Token + Position embeddings
    token_emb = layers.Embedding(vocab_size, embed_dim)(inputs)
    pos_emb = layers.Embedding(max_len, embed_dim)(tf.range(start=0, limit=max_len, delta=1))
    x = token_emb + pos_emb
    
    # Stack of encoder layers
    for _ in range(num_layers):
        x = BERTEncoder(embed_dim, num_heads, ff_dim)(x)
    
    # Pooler (for classification tasks)
    pooled = layers.Lambda(lambda x: x[:, 0, :])(x)  # [CLS] token
    pooled = layers.Dense(embed_dim, activation='tanh')(pooled)
    
    model = keras.Model(inputs, [x, pooled], name='BERT_Style')
    return model

print("BERT-style model architecture defined")
print("\nKey features:")
print("- Bidirectional attention (sees full context)")
print("- Pre-trained with MLM + NSP")
print("- Fine-tuned for specific tasks")

---

## 2. BERT: Bidirectional Encoder Representations from Transformers (COMPSCI 714 Lecture 8-1)

### Key Innovation

**BERT** = **B**idirectional **E**ncoder **R**epresentations from **T**ransformers

**Previous models:** Unidirectional (left-to-right)
- GPT: Only sees previous tokens
- ELMo: Shallow bidirectional

**BERT:** Deep **bidirectional** — sees both left and right context simultaneously.

### Architecture Details (COMPSCI 714)

| Model | Layers | Hidden Dim | Attention Heads | Parameters |
|-------|--------|-----------|-----------------|------------|
| **BERT-base** | 12 | 768 | 12 | 110M |
| **BERT-large** | 24 | 1024 | 16 | 340M |

**Trained on:**
- BooksCorpus (800 million words)
- English Wikipedia
- Pretrained with **64 TPU chips for 4 days** (expensive!)
- Finetuning is practical on a single GPU → **"Pretrain once, finetune many times"**

### Pre-training Tasks

**1. Masked Language Modeling (MLM)** — predict a random 15% of tokens:
- Replace input word with `[MASK]` — **80%** of the time
- Replace input word with a **random token** — **10%** of the time
- Leave input word **unchanged** but still predict it — **10%** of the time

**2. Next Sentence Prediction (NSP):**
- Given two sentences A and B, predict if B follows A
- Binary classification task (IsNextSentence / NotNextSentence)

### Fine-tuning BERT for Downstream Tasks

BERT can be fine-tuned for many tasks by adding a simple output layer:
- **Sentence pair classification** (e.g., MNLI, QQP): Use `[CLS]` token representation
- **Single sentence classification** (e.g., SST-2, CoLA): Use `[CLS]` token
- **Question answering** (e.g., SQuAD): Predict start/end span positions
- **Named entity recognition**: Token-level classification

**Result:** Finetuning BERT led to new state-of-the-art results on a broad range of tasks.

### Extensions of BERT

- **RoBERTa:** Train BERT longer + remove NSP → more compute, more data improves pretraining
- **DistilBERT:** 6 layers, 66M params — 97% of BERT-base performance, 4× faster
- **ALBERT:** 12M params — parameter sharing across layers

---

## 3. T5: Text-To-Text Transfer Transformer (COMPSCI 714 Lecture 8-1)

### Architecture: Encoder-Decoder

T5 uses the **full Transformer** (encoder + decoder), making it suitable for sequence-to-sequence tasks.

### Pretraining Objective: Span Corruption

Unlike BERT's token-level masking, T5 masks **contiguous spans** of tokens:

- **Input:** A corrupted version of a text sequence where random contiguous spans are masked and replaced with a single special token (e.g., `<X>`, `<Y>`)
- **Output:** The model generates the missing spans in order

**Example:**
```
Original:  "Thank you for inviting me to your party last week."
Input:     "Thank you <X> me to your party <Y> week."
Target:    "<X> for inviting <Y> last <Z>"
```

**Why Span Corruption?**
- Encourages **long-range understanding** and richer representations
- More suitable for **sequence-to-sequence tasks** (summarisation, translation)
- Matches the **unified text-to-text format**: all tasks are posed as text-in → text-out

### The Text-to-Text Framework

T5 frames **every NLP task** as text generation:

| Task | Input | Output |
|------|-------|--------|
| Translation | `"translate English to German: That is good."` | `"Das ist gut."` |
| Classification | `"cola sentence: The course is jumping well."` | `"not acceptable"` |
| Similarity | `"stsb sentence1: ... sentence2: ..."` | `"3.8"` |
| Summarisation | `"summarize: state authorities dispatched..."` | `"six people hospitalized..."` |

> 📖 **Reference:** Raffel et al. (2020), "Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer", JMLR

---

## Learning Progress Tracker

Use this section to track your learning progress for this lesson.

### Completion Status
- [ ] Lesson completed
- [ ] All code cells executed successfully
- [ ] Understood all key concepts
- [ ] Completed practice exercises (if any)

### Dates
- **First Completed:** ____/____/____
- **Last Reviewed:** ____/____/____
- **Next Review:** ____/____/____ (Recommended: 1 week, 1 month, 3 months)

### Understanding Level
Rate your understanding (1-5): _____ / 5

- 1 = Need to review completely
- 2 = Understood basics, need more practice
- 3 = Good understanding, minor gaps
- 4 = Strong understanding, can explain to others
- 5 = Mastered, can apply in projects

### Notes & Reflections
```
Write your notes here:
- What concepts were challenging?
- What was interesting or surprising?
- How can you apply this in projects?
- Questions to explore further?




```

### Key Concepts to Remember (I09)
- [ ] Pre-training and Fine-tuning Paradigm
- [ ] AI Scaling Law (model size, dataset size, compute)
- [ ] BERT: MLM masking strategy (80/10/10), NSP, BERT-base vs BERT-large
- [ ] T5: Span Corruption, text-to-text framework
- [ ] GPT: decoder-only, Causal Language Modeling
- [ ] Architecture comparison: Encoders vs Encoder-Decoders vs Decoders
- [ ] When to use each architecture

---